In [1]:
import pandas as pd

#cargamos solo 2020 para probar
sepe_2020 = pd.read_csv(
    "../data/raw/Contratos_por_municipios_2020_csv.csv",
    sep=";",                 # el SEPE usa punto y coma, no coma. Si no lo ponemos puede leer el archivo como una sola columna gigante
    encoding="ISO-8859-1",   # encoding típico de organismos públicos españoles, no UTF-8. Si no, nos lanzará un
                             # UnicodeDecodeError en cuanto encuentre una tilde 
    skiprows=1,              # la primera fila es un titulo repetido, no son datos
    dtype=str                # cargamos todo como texto por ahora, para inspeccionar
)

print ("Shape:" , sepe_2020.shape)
print()
print (sepe_2020.head(3))

Shape: (89446, 19)

  Código mes             mes Código de CA Comunidad Autónoma Codigo Provincia  \
0      202001  Enero de 2020            1          Andalucía                4   
1      202001  Enero de 2020            1          Andalucía                4   
2      202001  Enero de 2020            1          Andalucía                4   

  Provincia Codigo Municipio  Municipio Total Contratos  \
0   Almería            04001       Abla              68   
1   Almería            04002   Abrucena              71   
2   Almería            04003       Adra             474   

  Contratos iniciales indefinidos hombres  \
0                                       0   
1                                       0   
2                                      29   

  Contratos iniciales temporales hombres  \
0                                     31   
1                                     44   
2                                    260   

  Contratos convertidos en indefinidos hombres  \
0         

El encoding ha funcionado, se ven las tildes sin problemas.
Ahora a limpiar los nombres de columnas

In [2]:
print ("Columnas ANTES de limpiar:")
print (list(sepe_2020.columns))

#limpiamos espacios al principio y al final, y colapsamos espacios múltiples internos a uno solo
sepe_2020.columns = sepe_2020.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

print()
print("Columnas DESPUÉS de limpiar: ")
print(list(sepe_2020.columns))

Columnas ANTES de limpiar:
['Código mes ', 'mes', 'Código de CA', 'Comunidad Autónoma', 'Codigo Provincia', 'Provincia', 'Codigo Municipio', ' Municipio', 'Total Contratos', 'Contratos iniciales indefinidos hombres', 'Contratos iniciales temporales hombres', 'Contratos convertidos en indefinidos hombres', 'Contratos iniciales indefinidos mujeres', 'Contratos iniciales temporales mujeres', 'Contratos convertidos en indefinidos mujeres', 'Contratos  Agricultura', 'Contratos  Industria', 'Contratos Construcción', 'Contratos  Servicios']

Columnas DESPUÉS de limpiar: 
['Código mes', 'mes', 'Código de CA', 'Comunidad Autónoma', 'Codigo Provincia', 'Provincia', 'Codigo Municipio', 'Municipio', 'Total Contratos', 'Contratos iniciales indefinidos hombres', 'Contratos iniciales temporales hombres', 'Contratos convertidos en indefinidos hombres', 'Contratos iniciales indefinidos mujeres', 'Contratos iniciales temporales mujeres', 'Contratos convertidos en indefinidos mujeres', 'Contratos Agricul

Vamos a comprobar si hay valores "raros" (no numéricos) tiene

In [3]:
col = "Contratos Servicios"

# Buscamos valores que NO sean puramente numéricos
valores_unicos = sepe_2020[col].dropna().unique()
no_numericos = [v for v in valores_unicos if not v.strip().lstrip('-').isdigit()]

print("Valores no numéricos encontrados en 2020:", no_numericos)
print()
print("Total de valores únicos:", len(valores_unicos))

Valores no numéricos encontrados en 2020: []

Total de valores únicos: 2075


Ahora que sabemos que 2020 es "limpio", vamos a convertir Contratos Servicios a entero. 

In [4]:
# Convertimos a numérico sustituyendo "<5" por 0 cuando exista
# En 2020 no debe haber ningun, pero dejamos el .replace() preparado para los siguientes archivos

sepe_2020[col] = (
    sepe_2020[col]
    .str.strip() # Por si hay espacios sueltos alrededor del número
    .replace("<5", "0") # Tratamos los datos enmascarados como 0
    .astype(int) # Convierte de texto a número entero
)

print("Tipo de dato ahora:", sepe_2020[col].dtype)
print()
print("Total de ontratos en Servicios en TODO 2020:", sepe_2020[col].sum())
print()
print("Primeras filas para comprobar visualmente:")
print(sepe_2020[["Código mes", "mes", "Municipio", col]].head())

Tipo de dato ahora: int64

Total de ontratos en Servicios en TODO 2020: 9204699

Primeras filas para comprobar visualmente:
  Código mes            mes  Municipio  Contratos Servicios
0     202001  Enero de 2020       Abla                   51
1     202001  Enero de 2020   Abrucena                   58
2     202001  Enero de 2020       Adra                  260
3     202001  Enero de 2020  Albánchez                    5
4     202001  Enero de 2020  Alboloduy                    7


9.204.699 contratos en Servicios durante 2020 en toda España, y los valores por fila coinciden con los que vimos al inspeccionar el archivo crudo. 
Ahora pasamos a la agregación: total nacional por mes

In [7]:
agregado_2020 = (
    sepe_2020.groupby("Código mes")[col]    # Agrupamos todas las filas que comparten el mismo mes (todos los municipios de enero 2020 juntos, etc.)
    .sum()      # Dentro de cada grupo, suma la columna de Servicios
    .reset_index()    # Convierte el resultado de vuelta a un DataFrame normal con columnas
    .rename(columns={"Código mes": "año_mes", col: "contratos_servicios_total"})    # Nombres más claros para el dataset final
    .sort_values("año_mes") # Por si el orden no salía ya en cronológico
)

print(agregado_2020)
print()
print("Número de meses:", len(agregado_2020))

   año_mes  contratos_servicios_total
0   202001                    1136186
1   202002                    1081984
2   202003                     822605
3   202004                     326749
4   202005                     403450
5   202006                     714887
6   202007                    1071709
7   202008                     747043
8   202009                    1037476
9   202010                    1004948
10  202011                     857662

Número de meses: 11


Ahora repetimos el proceso con el resto de años

In [8]:
#cargamos 2021
sepe_2021 = pd.read_csv(
    "../data/raw/Contratos_por_municipios_2021_csv.csv",
    sep=";",                 # el SEPE usa punto y coma, no coma. Si no lo ponemos puede leer el archivo como una sola columna gigante
    encoding="ISO-8859-1",   # encoding típico de organismos públicos españoles, no UTF-8. Si no, nos lanzará un
                             # UnicodeDecodeError en cuanto encuentre una tilde 
    skiprows=1,              # la primera fila es un titulo repetido, no son datos
    dtype=str                # cargamos todo como texto por ahora, para inspeccionar
)

print ("Shape:" , sepe_2021.shape)
print()
print (sepe_2021.head(3))

Shape: (97608, 19)

  Código mes             mes Código de CA Comunidad Autónoma Codigo Provincia  \
0      202101  Enero de 2021            1          Andalucía                4   
1      202101  Enero de 2021            1          Andalucía                4   
2      202101  Enero de 2021            1          Andalucía                4   

  Provincia Codigo Municipio  Municipio Total Contratos  \
0   Almería            04001       Abla              21   
1   Almería            04002   Abrucena              92   
2   Almería            04003       Adra             456   

  Contratos iniciales indefinidos hombres  \
0                                       0   
1                                       2   
2                                      11   

  Contratos iniciales temporales hombres  \
0                                      9   
1                                     57   
2                                    248   

  Contratos convertidos en indefinidos hombres  \
0         

# Evaluación de SEPE como fuente complementaria

**Objetivo inicial:** enriquecer el análisis del mercado tech (dataset Kaggle) con datos
oficiales de contratación en España, usando el dataset público "Contratos por municipios"
del SEPE (datos.gob.es).

**Hallazgo:** la fuente solo desagrega por 4 categorías macro de actividad económica
(Agricultura, Industria, Construcción, Servicios). No existe una columna ni un desglose
a nivel CNAE 62 (programación/informática) en formato CSV abierto y reutilizable — ese
nivel de detalle solo aparece como gráficos dentro de informes PDF anuales del SEPE,
no estructurados.

**Decisión:** descartar la integración de SEPE en el dashboard principal, porque cruzar
"contratos en el sector Servicios" (que mezcla informática con hostelería, comercio,
banca, etc.) con datos de salarios tech internacionales generaría una conclusión poco
rigurosa.

**Alternativa identificada para una futura v2:** el INE publica la Encuesta de Población
Activa (EPA) con desagregación por la sección "J - Información y comunicaciones" de la
CNAE-2009, más cercana al sector tech aunque no exclusiva de él. No se ha integrado en
esta iteración del proyecto.

El código de exploración de abajo se conserva como evidencia del proceso de validación
de la fuente.